# AutoVideoBot - Kaggle batch image generator (FREE GPU)

Kaggle gives you **30 free GPU hours every week** (T4 x2, reset every Tuesday).
Kaggle cannot host a live server, so this notebook works in BATCH mode:

1. the bot wrote this notebook with **your prompts already inside it**
2. you run it here (2 minutes of clicking)
3. it saves every image into `output/images.zip`
4. you download that zip and drop it into `workspace/kaggle_out/`
5. run the same bot command again - it imports the images and continues

### Setup (once per notebook)
| Setting | Where | Value |
|---|---|---|
| Accelerator | right panel -> **Settings** | **GPU T4 x2** |
| Internet | right panel -> **Settings** | **ON** (needed to download the model) |

Then press **Run All**.


## 1. Check the GPU

In [ ]:
import torch
assert torch.cuda.is_available(), (
    "NO GPU! Open the right-hand Settings panel -> Accelerator -> GPU T4 x2, then Run all again."
)
print("GPU :", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), "GB")


## 2. Install the packages (about 2 minutes)

In [ ]:
!pip install -q diffusers transformers accelerate safetensors pillow
print("packages installed")


## 3. THE JOBS
These prompts were written by AutoVideoBot from your script.
You can edit them right here - just keep the `JOBS` list format the same.
Re-running the notebook SKIPS images that already exist, so it is always
safe to press Run all again after a disconnect.


In [ ]:
# >>>JOBS<<<  (generated by AutoVideoBot - do not edit)
MODEL_ID = "stabilityai/stable-diffusion-xl-base-1.0"
FILENAME_PREFIX = "s"
JOBS = []      # <- replaced with your real scenes when the bot writes this file


## 4. Generate every image

In [ ]:
import base64, io, json, os, time
import torch
from diffusers import AutoPipelineForText2Image
from PIL import Image

assert JOBS, "JOBS is empty - this notebook was not filled in by the bot.\n"\
               "Run the bot with:  python main.py images myvideo"

OUT = "images"
os.makedirs(OUT, exist_ok=True)
FAILED = []
MAX_PIXELS = 1024 * 1024
IS_TURBO = "turbo" in MODEL_ID.lower()

print("loading", MODEL_ID, "(downloads a few GB the first time) ...")
t0 = time.time()
pipe = AutoPipelineForText2Image.from_pretrained(MODEL_ID, torch_dtype=torch.float16)
pipe = pipe.to("cuda")
pipe.enable_attention_slicing()
pipe.enable_vae_slicing()
pipe.set_progress_bar_config(disable=True)
print(f"model ready in {time.time()-t0:.0f}s")

def fit(w, h):
    w, h = max(256, int(w)), max(256, int(h))
    if w * h > MAX_PIXELS:
        k = (MAX_PIXELS / (w * h)) ** 0.5
        w, h = int(w * k), int(h * k)
    return max(256, w - w % 8), max(256, h - h % 8)

done = 0
for job in JOBS:
    name = f"{FILENAME_PREFIX}{job['id'].lstrip('s')}"
    path = os.path.join(OUT, name + ".jpg")
    if os.path.exists(path):
        print(f"skip {job['id']} (already generated)")
        done += 1
        continue
    tw, th = max(8, round(int(job.get("width") or 1024) / 8) * 8), max(8, round(int(job.get("height") or 1024) / 8) * 8)
    gw, gh = fit(tw, th)
    steps = min(int(job.get("steps") or 30), 4) if IS_TURBO else int(job.get("steps") or 30)
    guidance = 0.0 if IS_TURBO else float(job.get("guidance_scale") or 7.0)
    for attempt in (1, 2):
        try:
            gen = None
            if job.get("seed") is not None:
                gen = torch.Generator(device="cuda").manual_seed(int(job["seed"]))
            with torch.inference_mode():
                out = pipe(prompt=job["prompt"], negative_prompt=job.get("negative_prompt") or None,
                           width=gw, height=gh, num_inference_steps=max(1, steps),
                           guidance_scale=guidance, generator=gen)
            img = out.images[0]
            if (img.width, img.height) != (tw, th):
                img = img.resize((tw, th), 1)
            img.convert("RGB").save(path, quality=94)
            done += 1
            print(f"[{done}/{len(JOBS)}] {job['id']}  {gw}x{gh} -> {tw}x{th}")
            break
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if attempt == 2:
                FAILED.append({"id": job["id"], "error": "out of VRAM"})
                print(f"FAILED {job['id']}: out of VRAM")
            else:
                gw, gh = fit(gw // 2, gh // 2)
                print(f"retrying {job['id']} smaller: {gw}x{gh}")
        except Exception as e:
            FAILED.append({"id": job["id"], "error": str(e)[:200]})
            print(f"FAILED {job['id']}: {e}")
            break
    torch.cuda.empty_cache()

json.dump(FAILED, open(os.path.join(OUT, "job_log.json"), "w"), indent=2)
print(f"\nfinished: {done}/{len(JOBS)} image(s), {len(FAILED)} failed")


## 5. Package the images for download

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/images", "zip", "images")
print("created /kaggle/working/images.zip")
print()
print("NEXT STEPS:")
print("  1. open the Data / Output panel on the right")
print("  2. download  images.zip")
print("  3. move it into your bot folder:  workspace/kaggle_out/")
print("  4. run the same bot command again - it imports the images and continues")


## Troubleshooting
| Symptom | Fix |
|---|---|
| `assert NO GPU` | Settings panel -> Accelerator: **GPU T4 x2** |
| Out of memory | The cell retries at a smaller size automatically. Or lower `steps` in the JOBS cell |
| Notebook stopped at 90 minutes | Re-run all; already-saved images are skipped |
| Some images failed | Check `images/job_log.json`. Re-run, or lower `steps` |

Kaggle gives 30 free GPU hours per week, reset every Tuesday.
